# Raw Data Exploration

This notebook explores the raw data extracted by the ETL pipeline.

## Data Files

| File | Description |
|------|-------------|
| `artifacts/raw_users.parquet` | User accounts from Aurora DB |
| `artifacts/raw_insertions.parquet` | Listing insertions with JSON payloads |

## ETL Pipeline

The ETL process (`src/data/etl.py`) extracts data from Aurora PostgreSQL:
- **Chunking**: Processes large tables in configurable chunks
- **Checkpointing**: Resumes from last successful chunk
- **JSON Parsing**: Extracts structured fields from `listing_json` and `selected_bundle_json`

Run ETL: `python src/cli.py extract-data` or `make etl`

In [1]:
import polars as pl
import os

# Set display options
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

polars.config.Config

## 1. Load Raw Data

In [2]:
users_path = "../artifacts/raw_users.parquet"
insertions_path = "../artifacts/raw_insertions.parquet"

if os.path.exists(users_path):
    df_users = pl.read_parquet(users_path)
    print(f"Users: {df_users.shape}")
else:
    print("Users file not found.")

if os.path.exists(insertions_path):
    df_insertions = pl.read_parquet(insertions_path)
    print(f"Insertions: {df_insertions.shape}")
else:
    print("Insertions file not found.")

Users: (451764, 4)
Insertions: (237695, 12)


## 2. Explore Users

In [3]:
if 'df_users' in locals():
    display(df_users.head())

user_id,owner_id,created_at,contact_emails
i32,str,"datetime[μs, UTC]",str
1,"""2215010""",2020-12-17 15:17:42.074491 UTC,"""regula.stutz@stutz-mechanik.ch"""
2,"""275820823""",2020-12-17 15:18:18.951771 UTC,"""dk@kunzarchitektur.ch;info@kunzarchitektur.ch;mela…"
3,"""2161140""",2020-12-17 15:18:31.712636 UTC,"""insertion-funnel-smoke-test@homegate.ch;insertion-…"
5,"""890371""",2020-12-17 15:19:20.843158 UTC,"""mabol@bluemail.ch"""
6,"""537853""",2020-12-17 15:19:23.732662 UTC,"""k.huwiler@huwiler-immobilien.ch;speedtechnik@bluew…"


## 3. Explore Insertions

In [4]:
if 'df_insertions' in locals():
    display(df_insertions.head())

object_reference,platform,user_id,user_ip_address,listing_json,fraud_flag,auto_approval_criteria_json,first_published_date,customer_segment,selected_bundle_json,listing_created_at,submission_at
str,str,i32,str,str,"datetime[μs, UTC]",str,"datetime[μs, UTC]",str,str,"datetime[μs, UTC]","datetime[μs, UTC]"
"""hgonif#wznna.qq6xs##""","""homegate""",6719,"""162.158.148.228""","""{""id"": ""3002320764"", ""meta"": {""source"": ""INSERTION…",null,"""{""meta"": {""seonFraudScore"": 0}, ""criteria"": {""seon…",2023-01-01 07:36:39.930 UTC,null,"""{""id"": ""BUNDLE_TFP-1228_DCH_BASIC"", ""tier"": ""basic…",2023-01-01 07:34:26.394887 UTC,2023-01-01 07:36:27.382 UTC
"""hgonif#hl6iv.bm4k6##""","""homegate""",723096,"""185.110.74.33""","""{""id"": ""3002320767"", ""meta"": {""schemaVersion"": ""1.…",null,"""{""meta"": {""seonFraudScore"": 5}, ""criteria"": {""seon…",2023-01-01 09:05:31.853 UTC,"""tenant""","""{""id"": ""BUNDLE_T2T-EXP-PKP-2575_vB_ZH_BASIC"", ""tie…",2023-01-01 08:14:00.688451 UTC,2023-01-01 09:05:22.750 UTC
"""hgonif#moufi.9ub4j##""","""homegate""",46391,"""162.158.148.195""","""{""id"": ""3002320770"", ""meta"": {""source"": ""INSERTION…",null,"""{""meta"": {""seonFraudScore"": 0}, ""criteria"": {""seon…",2023-01-01 10:09:37.981 UTC,null,"""{""id"": ""BUNDLE_TFP-1228_ZH_TOP"", ""tier"": ""top"", ""p…",2022-12-27 15:01:20.231117 UTC,2023-01-01 10:09:29.548 UTC
"""hgonif#6umnv.cakxs##""","""homegate""",723100,"""62.202.182.38""","""{""id"": ""3002320771"", ""meta"": {""schemaVersion"": ""1.…",null,"""{""meta"": {""seonFraudScore"": 0}, ""criteria"": {""seon…",2023-01-01 10:13:27.263 UTC,"""tenant""","""{""id"": ""BUNDLE_T2T-EXP-PKP-2575_vB_ZG_BASIC"", ""tie…",2023-01-01 09:44:30.940867 UTC,2023-01-01 10:13:19.756 UTC
"""hgonif#3us76.0f7zo##""","""homegate""",140429,"""178.196.52.232""","""{""id"": """", ""meta"": {""schemaVersion"": ""1.12.0"", ""is…",null,"""{""meta"": {""seonFraudScore"": 37}, ""criteria"": {""seo…",null,"""owner""","""{""id"": ""BUNDLE_TFP-1228_WCH_TOP"", ""tier"": ""top"", ""…",2022-12-17 14:06:59.018623 UTC,2023-01-01 10:35:39.222 UTC


### 3.1 JSON Parsing Example
The `listing` and `selected_bundle` columns are stored as JSON strings. Here is how to parse them to extract specific fields.

In [7]:
if 'df_insertions' in locals():
    # Example: Extract City, Price, and Title
    # We use Python's json module with map_elements for compatibility across Polars versions
    import json
    
    def safe_extract(json_str, *keys):
        """Safely extract nested keys from JSON string"""
        try:
            data = json.loads(json_str)
            for key in keys:
                data = data.get(key, {})
                if data is None or data == {}:
                    return None
            return data
        except:
            return None
    
    df_parsed = df_insertions.select([
        pl.col("object_reference"),
        pl.col("submission_at"),
        
        # Extract City
        pl.col("listing_json").map_elements(
            lambda x: safe_extract(x, "address", "locality"), 
            return_dtype=pl.Utf8
        ).alias("city"),
        
        # Extract Rent Gross Price
        pl.col("listing_json").map_elements(
            lambda x: safe_extract(x, "prices", "rent", "gross"), 
            return_dtype=pl.Float64
        ).alias("rent_gross"),
        
        # Extract Title (German)
        pl.col("listing_json").map_elements(
            lambda x: safe_extract(x, "localization", "de", "text", "title"), 
            return_dtype=pl.Utf8
        ).alias("title_de")
    ])
    
    display(df_parsed.head())

object_reference,submission_at,city,rent_gross,title_de
str,"datetime[μs, UTC]",str,f64,str
"""hgonif#wznna.qq6xs##""",2023-01-01 07:36:27.382 UTC,"""Ettingen""",780.0,"""Gewerberaum für Büro, Physio, Massage, Nagelstudio…"
"""hgonif#hl6iv.bm4k6##""",2023-01-01 09:05:22.750 UTC,"""Zürich""",2291.0,"""Einmalige Gelegenheit! Leben auf einem Wein- und O…"
"""hgonif#moufi.9ub4j##""",2023-01-01 10:09:29.548 UTC,"""Zürich""",3950.0,"""Aussergewöhnliche 3.5-Z._Wohnung in einem sanierte…"
"""hgonif#6umnv.cakxs##""",2023-01-01 10:13:19.756 UTC,"""Hünenberg""",4800.0,"""Büro in Hünenberg"""
"""hgonif#3us76.0f7zo##""",2023-01-01 10:35:39.222 UTC,"""Giez""",2030.0,"""Maison individuelle à Giez"""


### 3.2 Check Customer Segments

In [6]:
if 'df_insertions' in locals():
    print(df_insertions["customer_segment"].value_counts())

shape: (5, 2)
┌──────────────────┬────────┐
│ customer_segment ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ null             ┆ 35105  │
│ professionalzzz> ┆ 1      │
│ tenant           ┆ 67793  │
│ professional     ┆ 32530  │
│ owner            ┆ 102266 │
└──────────────────┴────────┘
